In [6]:
import gradio as gr
import cv2
import numpy as np
import pandas as pd
from skimage.util import random_noise
from scipy.ndimage import laplace, sobel
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- Helper: Histogram Plot ---
def plot_histogram(img):
    if img is None: return None
    plt.figure(figsize=(3, 2))
    plt.hist(img.flatten(), bins=64, color='gray', alpha=0.7)
    plt.title("Intensity Distribution", fontsize=8)
    plt.xticks(fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout(); plt.savefig("hist_temp.png"); plt.close()
    return "hist_temp.png"

# --- TAB 1: Restoration & Enhancement ---
def dynamic_restoration(src, n_type, n_lvl, p_method, gamma, f_type, k_size, d0):
    if src is None:
        return [None]*4 + ["Waiting for Image..."]

    src = cv2.resize(src, (350, 350))
    gray_orig = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY) if len(src.shape) == 3 else src

    # 1. Optional Degradation (Noise Injection)
    noisy = gray_orig.copy()

    if n_type == "Salt & Pepper":
        prob = n_lvl if n_lvl <= 1.0 else n_lvl / 100.0
        noisy = random_noise(noisy, mode="s&p", amount=prob)
        noisy = (noisy * 255).astype(np.uint8)

    elif n_type == "Gaussian":
      img = gray_orig.astype(np.float32) / 255.0
      rng = np.random.default_rng(0)
      noise = rng.normal(0, n_lvl, img.shape).astype(np.float32)
      noisy = np.clip(img + noise, 0, 1)
      noisy = (noisy * 255).astype(np.uint8)

    elif n_type == "Periodic":
        img = gray_orig.astype(np.float32) / 255.0
        rows, cols = img.shape
        x = np.arange(cols)
        noise = n_lvl * np.sin(2 * np.pi * x / 20)
        noise_2d = np.tile(noise, (rows, 1)).astype(np.float32)

        noisy = np.clip(img + noise_2d, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)

    elif n_type == "Rayleigh":
        img = gray_orig.astype(np.float32) / 255
        noise = np.random.rayleigh(n_lvl, img.shape)
        noisy = img + noise
        noisy = np.clip(noisy, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)

    elif n_type == "Uniform":
      img = gray_orig.astype(np.float32) / 255.0
      rng = np.random.default_rng(0)
      noise = rng.uniform(0, n_lvl, img.shape).astype(np.float32)
      noisy = np.clip(img + noise, 0, 1)
      noisy = (noisy * 255).astype(np.uint8)



    # 2. Intensity Transformations (Enhancement)

    work = noisy.astype(np.float32) / 255.0

    if p_method == "Negative":
        work = 1.0 - work

    elif p_method == "Log":
        work = np.log1p(work)
        work = work / (work.max() + 1e-12)

    elif p_method == "Gamma":
        work = np.power(work, gamma)

    elif p_method == "Histogram Equalization":
        temp = (work * 255).astype(np.uint8)

        # Step 1: Calculate the histogram
        hist, bins = np.histogram(temp.flatten(), bins=256, range=(0, 256))

        # Step 2: Calculate the Cumulative Distribution Function (CDF)
        cdf = hist.cumsum()

        # Step 3: Normalise the CDF to be in the range [0, 255]
        cdf_norm = (cdf - cdf.min()) * 255 / (cdf.max() - cdf.min())
        cdf_norm = cdf_norm.astype(np.uint8)

        # Step 4: Map the original image pixels to the new values
        temp_eq = cdf_norm[temp]

        work = temp_eq.astype(np.float32) / 255.0

    processed = np.clip(work * 255, 0, 255).astype(np.uint8)

    # 3. Filtering (Restoration)
    restored = processed.copy()
    if f_type != "None":
        k = int(k_size) | 1
        if f_type == "Mean":
            restored = cv2.blur(processed, (k, k)) # Example implemented

        elif f_type == "Median":
            restored = cv2.medianBlur(processed, k)

        elif f_type == "Laplacian":
            lap = laplace(processed.astype(float))
            # 2. Subtract Laplacian from the original image to sharpen it
            restored = np.clip(processed - lap, 0, 255).astype(np.uint8)

        elif f_type == "Sobel":
          # 1. Calculate horizontal gradient (axis=0)
          sx = cv2.Sobel(processed, cv2.CV_64F, 1, 0, ksize=k)
          # 1. Calculate vertical gradient (axis=0)
          sy = cv2.Sobel(processed, cv2.CV_64F, 0, 1, ksize=k)
          # Calculate the magnitude of the edges
          restored = np.clip(np.hypot(sx, sy), 0, 255).astype(np.uint8)

        elif f_type in ["Low Pass", "High Pass", "Notch"]:

            # Implement Frequency Domain Filtering (FFT)
            img_float = processed.astype(np.float32) / 255.0
            F = np.fft.fft2(img_float)
            F_shift = np.fft.fftshift(F)

            #Compute magnitude spectrum for visualization
            magnitude_spectrum = np.log(1 + np.abs(F_shift))

            #Get image dimensions and center
            rows, cols = processed.shape
            crow, ccol = rows // 2, cols // 2

            #Create distance matrix from center
            U, V = np.ogrid[:rows, :cols]
            D = np.sqrt((U - crow)**2 + (V - ccol)**2)

            #Apply Gaussian Low Pass Filter
            if f_type == "Low Pass":
                H = np.exp(-(D**2) / (2 * (d0**2)))

            #Apply Gaussian High Pass Filter
            elif f_type == "High Pass":
               H = 1 - np.exp(-(D**2) / (2 * (d0**2)))

            #Apply Ideal Notch filter
            elif f_type == "Notch":
                H = np.ones((rows, cols), dtype=np.float32)
                notches = [
                    (crow, ccol + int(d0)),
                    (crow, ccol - int(d0))
                ]

                radius = 5
                for (u0, v0) in notches:
                    Dn = np.sqrt((U - u0)**2 + (V - v0)**2)
                    H[Dn <= radius] = 0

            #Apply filter in frequency domain
            G = F_shift * H

            #Apply inverse FFT to return to image
            img_back = np.real(np.fft.ifft2(np.fft.ifftshift(G)))

            #Normalize image values to (0–255)
            img_back = (img_back - img_back.min()) / (img_back.max() - img_back.min() + 1e-12)
            restored = (img_back * 255).astype(np.uint8)

    # 4. Quantitative Metrics
    mse_val = np.mean((gray_orig.astype(np.float32) - restored.astype(np.float32))**2)
    psnr = 100 if mse_val == 0 else 20 * np.log10(255.0 / np.sqrt(mse_val))

    return restored, cv2.absdiff(gray_orig, restored), plot_histogram(restored), f"MSE: {mse_val:.2f} | PSNR: {psnr:.2f} dB"

# --- TAB 2: Segmentation & Morphology ---
def dynamic_segmentation(img, seg_meth, threshold, morph_op, se_shape, se_size, class_label):
    if img is None: return [None]*3
    img = cv2.resize(img, (350, 350))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

    # 1. Segmentation
    if seg_meth == "Global":
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    elif seg_meth == "Adaptive":
        block_size = 11
        C = 2
        binary = cv2.adaptiveThreshold(gray,255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, block_size,C)

    elif seg_meth == "Otsu":
        _, binary = cv2.threshold(gray,0,255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    else:
        binary = np.zeros_like(gray)

    # 2. Morphology
    s_size = int(se_size)

    if se_shape == 'Square':
      se = np.ones((s_size, s_size), np.uint8)

    elif se_shape=='Cross':
      se = cv2.getStructuringElement(cv2.MORPH_CROSS,(s_size,s_size))

    elif se_shape=='Disk':
      se= cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(s_size,s_size))



    m_out = binary.copy()

    if morph_op == "Erosion":
      m_out = cv2.erode(binary, se)

    elif morph_op=='Dilation':
      m_out=cv2.dilate(binary,se)

    elif morph_op=='Opening':
      m_out=cv2.morphologyEx(binary,cv2.MORPH_OPEN,se)

    elif morph_op=='Closing':
      m_out=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,se)

    elif morph_op=='Boundary Extraction':
      eroded= cv2.erode(binary,se)
      m_out=cv2.subtract(binary,eroded)



    # 3. Feature Extraction
    contours, _ = cv2.findContours(m_out, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    feats = []
    for i, c in enumerate(contours):
        area = cv2.contourArea(c)
        if area < 50: continue
        perim = cv2.arcLength(c, True)
        circ = ( 4 * np.pi * area ) / (perim **2) if perim > 0 else 0

        # compactness
        compactness = (perim**2) / area if area > 0 else 0

        # Moments & HuMoments
        M = cv2.moments(c)
        hu = cv2.HuMoments(M).flatten()

        # Solidity
        convexHull = cv2.convexHull(c)
        hull_area = cv2.contourArea(convexHull)
        solidity = area / hull_area if hull_area > 0 else 0

        feats.append({"Area": round(area,1),
                      "Perimeter": round(perim,1),
                      "Circularity": round(circ,3),
                      "Compactness": round(compactness, 3),
                      "Hu_1": hu[0],
                      "Solidity": round(solidity, 3),
                      "Class": class_label})



    df = pd.DataFrame(feats)
    df.to_csv("current_sample.csv", index=False)
    return m_out, df, "current_sample.csv"




# --- TAB 3: Batch Analysis (Logic remains for evaluation) ---
def run_classification_analysis(train_file, test_file):

    try:
        df_tr = pd.read_csv(train_file.name) if train_file.name.endswith('.csv') else pd.read_excel(train_file.name)
        df_ts = pd.read_csv(test_file.name) if test_file.name.endswith('.csv') else pd.read_excel(test_file.name)

        cols = ['Area', 'Perimeter', 'Circularity', 'Compactness', 'Hu_1', 'Solidity']

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(df_tr[cols].values)
        X_ts_scaled = scaler.transform(df_ts[cols].values)

        pca = PCA(n_components=2)

        X_tr_pca = pca.fit_transform(X_tr_scaled)
        X_ts_pca = pca.transform(X_ts_scaled)

        # Predictions
        preds = []

        for vec in X_ts_pca:

            dists = [np.linalg.norm(vec - tr_vec) for tr_vec in X_tr_pca]

            preds.append(
                df_tr.iloc[np.argmin(dists)]['Class']
                if 'Class' in df_tr.columns
                else "Label Missing"
            )

        # Colors for each class
        colors = {
            'Apple': 'blue',
            'Banana': 'green',
            'Apple_with_leaf': 'orange'
        }

        # Plot
        plt.figure(figsize=(8,6))

        # Training points by class
        for cls in df_tr['Class'].unique():

            idx = df_tr['Class'] == cls

            plt.scatter(
                X_tr_pca[idx, 0],
                X_tr_pca[idx, 1],
                c=colors.get(cls, 'black'),
                label=cls,
                alpha=0.7,
                s=80
            )

        # Test points
        for i, pred in enumerate(preds):

            plt.scatter(
                X_ts_pca[i, 0],
                X_ts_pca[i, 1],
                c=colors.get(pred, 'red'),
                marker='x',
                s=100
            )

            # Prediction label
            plt.annotate(
                pred,
                (X_ts_pca[i, 0], X_ts_pca[i, 1]),
                xytext=(8, 8),
                textcoords='offset points',
                fontsize=7,
                color=colors.get(pred, 'red'),
                weight='bold',
                alpha=0.8
            )

        # Train labels
        for i, txt in enumerate(df_tr['Class']):

            plt.annotate(
                txt,
                (X_tr_pca[i, 0], X_tr_pca[i, 1]),
                xytext=(5, 5),
                textcoords='offset points',
                fontsize=8,
                color='black'
            )

        plt.title("PCA Cluster Visualization with Labels")

        plt.legend()

        plt.grid(True, alpha=0.2)

        plt.margins(0.3)

        plt.tight_layout()

        plt.savefig("pca_plot.png")

        plt.close()

        df_ts['Prediction'] = preds

        return df_ts, "pca_plot.png", f"Success: {len(df_ts)} samples predicted."

    except Exception as e:

        return None, None, f"Error: {str(e)}"
# --- UI Layout  ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DIAPF: Dynamic Image Analysis & Processing Framework")
    with gr.Tabs():
        with gr.TabItem("1. Restoration & Enhancement"):
            with gr.Row():
                with gr.Column(scale=2):
                    src_img = gr.Image(label="Source Image", type="numpy", height=350, width=350)
                    mse_out = gr.Textbox(label="Metrics (MSE & PSNR)")
                with gr.Column(scale=3, variant="panel"):
                    gr.Markdown("### Phase 1 Control")
                    with gr.Accordion("Noise Injection", open=False):
                        n_t = gr.Dropdown(["None", "Salt & Pepper", "Gaussian","Periodic","Rayleigh", "Uniform"], label="Noise Type", value="None"); n_l = gr.Slider(0, 0.2, 0.05, label="Intensity")
                    with gr.Accordion("Filters & Enhancement", open=True):
                        p_m = gr.Radio(["None", "Negative", "Log", "Gamma", "Histogram Equalization"], label="Transformation", value="None"); gam = gr.Slider(0.1, 5.0, 1.0, label="Gamma")
                        f_t = gr.Dropdown(["None", "Mean", "Median", "Laplacian", "Sobel", "Low Pass", "High Pass", "Notch"], label="Filter", value="None"); k_s = gr.Slider(1, 15, 3, step=2, label="Kernel Size"); d_0 = gr.Slider(1, 200, 50, label="D0 Cutoff")
                with gr.Column(scale=2):
                    res_out = gr.Image(label="Restored", height=350, width=350); dif_out = gr.Image(label="Diff Map", height=200, width=350); his_out = gr.Image(label="Histogram", height=150, width=350)
            t1_in = [src_img, n_t, n_l, p_m, gam, f_t, k_s, d_0]; [i.change(dynamic_restoration, t1_in, [res_out, dif_out, his_out, mse_out]) for i in t1_in]

        with gr.TabItem("2. Segmentation & Morphology"):
            with gr.Row():
                with gr.Column(scale=1):
                    seg_src = gr.Image(label="Input Image", height=350, width=350)
                    gr.Markdown("### Phase 2: Structural Extraction")
                    sm = gr.Radio(["Global", "Adaptive", "Otsu"], label="Method", value="Global"); st = gr.Slider(0, 255, 127, label="Threshold")
                    mo = gr.Dropdown(["None", "Erosion", "Dilation", "Opening", "Closing", "Boundary Extraction"], label="Operation", value="None")
                    sh = gr.Radio(["Square", "Cross", "Disk"], label="Shape", value="Square"); sz = gr.Slider(3, 15, 3, step=2, label="Size")
                with gr.Column(scale=1):
                    bin_out = gr.Image(label="Binary Mask", height=350, width=350); feat_out = gr.DataFrame(label="Features Vector", interactive=True)
                    gr.Markdown("> **IMPORTANT NOTE:** Use the 'Class Name' field below *only* for Training References. Leave it blank for Test Samples.")
                    c_lbl = gr.Textbox(label="Class Name (Required for Training Only)", placeholder="e.g. Coin_A", interactive=True)
                    file_out = gr.File(label="Download CSV Record")
            t2_in = [seg_src, sm, st, mo, sh, sz, c_lbl]; [i.change(dynamic_segmentation, t2_in, [bin_out, feat_out, file_out]) for i in t2_in]

        with gr.TabItem("3. Classification & PCA"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Phase 3: Analytical Analysis")
                    ftr = gr.File(label="Upload Training Reference (CSV)"); fts = gr.File(label="Upload Final Test Samples (CSV)")
                    bc = gr.Button("Run Analysis", variant="primary")
                with gr.Column():
                    pp = gr.Image(label="PCA Mapping"); cr = gr.Textbox(label="Status Report")
            ct = gr.DataFrame(label="Final Results Table"); bc.click(run_classification_analysis, [ftr, fts], [ct, pp, cr])

demo.launch()

/tmp/ipykernel_35008/4101635974.py:391: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2bd08c1d18d568747f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
